In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
from vinecopulas.marginals import best_fit_distribution
from vinecopulas.vinecopula import fit_vinecop

In [ ]:
###### Candidate copula families and their numeric IDs used by vinecopulas
copula_names = {
    1: "Gaussian", 2: "Gumbel0", 3: "Gumbel90", 4: "Gumbel180", 5: "Gumbel270",
    6: "Clayton0", 7: "Clayton90", 8: "Clayton180", 9: "Clayton270",
    10: "Frank", 11: "Joe0", 12: "Joe90", 13: "Joe180", 14: "Joe270", 15: "Student",
}

def print_last_vine_tree(a, p, c):
    """
    Prints a summary of the deepest (last) tree in a fitted vine copula.

    Arguments:
        a (np.ndarray): Triangular vine structure matrix (d x d), where
                        each column encodes the variable ordering of one
                        pair-copula edge.
        p (np.ndarray): Parameter matrix (d-1 x d-1); p[i, k] holds the
                        fitted parameter(s) for tree i, edge k.
        c (np.ndarray): Copula-family index matrix (d-1 x d-1); c[i, k]
                        is the integer ID looked up in `copula_names`.

    Returns:
        None  (output is printed to stdout)
    """
    dimen = a.shape[0]
    i = dimen - 2 

    print(f"** Tree:  {i + 1}")
    for k in range(dimen - 1 - i):
        ak = a[:, k] 
        akn = [int(ak[-1 - k]), int(ak[i])]

        conditioning_set = list((ak.astype(int)[:i])[::-1])
        cond_str = ",".join(map(str, conditioning_set))
        node_str = f"{akn[0]},{akn[1]} | {cond_str}"

        cop_id = c[i, k]
        name = copula_names.get(cop_id, f"Copula_{cop_id}")
        params = p[i, k]

        print(f"{node_str: <15} --->  {name} : parameters =  {params}")

In [ ]:
def get_dataset(datapath, class_col, short_items=None):
    """
    Load a CSV dataset, impute missing values, and split by binary class label.

    Steps:
      1. Read the CSV at `datapath`.
      2. Drop housekeeping columns (level_0, index, Unnamed: 0) if present.
      3. Report the overall missing-value rate.
      4. Fill remaining NaNs with per-column medians.
      5. Separate rows where `class_col == 1` from rows where `class_col == 0`.
      6. Optionally rename columns using the `short_items` mapping.

    Arguments:
        datapath (str):   Path to the CSV file.
        class_col (str):  Name of the binary class column (0/1).
        short_items (dict | None): Optional column-rename mapping
                                   (long name -> short name).

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: (df_class1, df_class0), each
        without the class column.
    """
    df = pd.read_csv(datapath)
    df = df.drop(columns=['level_0', 'index', 'Unnamed: 0'], errors='ignore')
    print(f"{datapath} | missing rate: {round(df.isna().sum().sum() / df.size, 4)}")

    # Impute missing values with the median of each numeric column
    df = df.fillna(df.median(numeric_only=True))

    # Split into positive (class 1) and negative (class 0) cohorts
    df_1 = df[df[class_col] == 1].drop(columns=class_col)
    df_0 = df[df[class_col] == 0].drop(columns=class_col)

    if short_items is not None:
        # Rename columns to their shorter display names
        df_1 = df_1.rename(columns=short_items)
        df_0 = df_0.rename(columns=short_items)

    print(f"{datapath} | class 1:", df_1.shape)
    print(f"{datapath} | class 0:", df_0.shape)
    return df_1, df_0


def get_u(df):
    """
    Transform each column of a DataFrame to uniform [0, 1] marginals.

    Arguments:
        df (pd.DataFrame): Raw feature matrix (n_samples x n_features).

    Returns:
        np.ndarray: Array of shape (n_samples, n_features) with values in (0, 1).
    """
    x = np.array(df)
    u = x.copy()

    for i in range(len(df.columns)):
        namedist = 'dist' + str(i)
        # Fit the best marginal distribution for variable i
        locals()[namedist] = best_fit_distribution(x[:, i])
        # Apply the CDF to obtain uniform pseudo-observations
        u[:, i] = locals()[namedist][0].cdf(u[:, i], *locals()[namedist][1])
    return u


def fit_and_save_vine(
    u, cohort_name, out_dir="vine_outputs",
    cops=None, vine=None, extra_meta=None
):
    """
    Fit a vine copula model to uniform data and persist the result to disk.

    The fitted model (structure matrix M, parameter matrix P, copula-family
    matrix C) is bundled together with metadata and serialised as a pickle
    file named `vine_{cohort_name}_{vine}.pkl` inside `out_dir`.

    Arguments:
        u (np.ndarray):       Uniform pseudo-observations, shape (n, d).
        cohort_name (str):    Label for this cohort (used in the filename).
        out_dir (str):        Directory where the pickle file is saved.
                              Created automatically if it does not exist.
        cops (list[int] | None): Candidate copula family IDs to consider.
                              Defaults to all 15 families (1–15).
        vine (str | None):    Vine type passed to `fit_vinecop`
                              (e.g. 'C' for C-vine, 'R' for R-vine).
        extra_meta (dict | None): Additional key-value pairs merged into
                              the saved result dict (e.g. datapath, class_value).

    Returns:
        tuple: (M, P, C, save_path)
            M (np.ndarray):   Vine structure matrix.
            P (np.ndarray):   Fitted copula parameter matrix.
            C (np.ndarray):   Fitted copula family index matrix.
            save_path (str):  Absolute path of the saved pickle file.
    """
    if cops is None:
        cops = list(range(1, 16))  # use all 15 candidate families by default

    os.makedirs(out_dir, exist_ok=True)

    # Fit the vine copula; returns structure (M), parameters (P), families (C)
    M, P, C = fit_vinecop(u, cops, vine=vine)

    # Bundle model arrays with provenance metadata for reproducibility
    result = {
        "cohort": cohort_name,
        "vine": vine,
        "copulas": cops,
        "M": M,
        "P": P,
        "C": C,
        "n": u.shape[0],  # number of observations
        "d": u.shape[1],  # number of dimensions / variables
    }
    if extra_meta:
        result.update(extra_meta)  # merge any caller-supplied metadata

    save_path = os.path.join(out_dir, f"vine_{cohort_name}_{vine}.pkl")
    with open(save_path, "wb") as f:
        pickle.dump(result, f)

    print(f"Saved: {save_path}")
    return M, P, C, save_path


def run_dataset(datapath, class_col=None, dataset_tag=None, out_dir=None, vine=None, cops=None):
    """
    End-to-end pipeline: load data, transform to uniforms, fit vine copula(s),
    and save the results.

    Behaviour depends on `dataset_tag`:
      - 'combined' / 'combined_balanced_gender' / 'combined_balanced_gender_class':
            Treat the entire dataset as a single cohort. Fit one vine copula
            and return a dict with key 'combined'.
      - Any other tag:
            Split the dataset by `class_col` into two cohorts (class 1 and
            class 0). Fit a separate vine copula for each and return a dict
            with keys 'class1' and 'class0'.

    Arguments:
        datapath (str):       Path to the input CSV file.
        class_col (str | None): Column name used for binary splitting
                              (required for the split-cohort path).
        dataset_tag (str | None): Short identifier for this dataset run;
                              also used in output filenames.
        out_dir (str | None): Directory for saved vine pickle files.
        vine (str | None):    Vine type ('C', 'R', etc.).
        cops (list[int] | None): Candidate copula family IDs.

    Returns:
        dict: For the combined path: {'combined': {'M', 'P', 'C', 'file'}}.
              For the split path:    {'class1':   {'M', 'P', 'C', 'file'},
                                      'class0':   {'M', 'P', 'C', 'file'}}.
    """
    if dataset_tag == 'combined' or 'combined_balanced_gender':
        # ── Combined cohort path ────────────────────────────────────────────
        df = pd.read_csv(datapath)  # load data
        # Drop housekeeping / ID columns that carry no feature information
        df = df.drop(columns=['level_0', 'index', 'Unnamed: 0', 'person_id'], errors='ignore')
        df = df.fillna(df.median(numeric_only=True))  # median imputation
        print(f"{datapath} | combined:", df.shape)
        print(df.isna().sum().sum() / df.size)  # sanity-check: should be 0 after imputation

        # Transform all columns to uniform marginals before copula fitting
        u = get_u(df)

        M, P, C, p = fit_and_save_vine(
            u, cohort_name=f"{dataset_tag}",
            out_dir=out_dir, vine=vine,
            cops=cops,
            extra_meta={"datapath": datapath})

        return {"combined": {"M": M, "P": P, "C": C, "file": p}}

    else:
        # ── Split-cohort path (class 1 vs class 0) ──────────────────────────
        df_1, df_0 = get_dataset(datapath, class_col, short_items=short_items)

        # Transform each cohort independently to uniform pseudo-observations
        u_1 = get_u(df_1)
        u_0 = get_u(df_0)

        # Fit and persist a vine copula for each cohort
        M1, P1, C1, p1 = fit_and_save_vine(
            u_1, cohort_name=f"{dataset_tag}_class1",
            out_dir=out_dir, vine=vine,
            cops=cops,
            extra_meta={"datapath": datapath, "class_value": 1})
        M0, P0, C0, p0 = fit_and_save_vine(
            u_0, cohort_name=f"{dataset_tag}_class0",
            out_dir=out_dir, vine=vine,
            cops=cops,
            extra_meta={"datapath": datapath, "class_value": 0})

        return {
            "class1": {"M": M1, "P": P1, "C": C1, "file": p1},
            "class0": {"M": M0, "P": P0, "C": C0, "file": p0},
        }

In [ ]:
save_path = 'vine_outputs/4_distribition/balanced_new'
cop_4 = [1, 6, 10, 15] # only 1: "Gaussian", 6: "Clayton0", 10: "Frank", 15: "Student" are considered
i = 'R' # 'C'
###### The dataset contains both heart failure patients (label 1) and non-CVD patients (label 0) 
# male_res = run_dataset("hf_icd_male.csv", class_col="class", out_dir=save_path, dataset_tag="male", vine=i, cops = cop_4)
# female_res = run_dataset("hf_icd_female.csv", class_col="class", out_dir=save_path, dataset_tag="female", vine=i, cops = cop_4)
com_res = run_dataset("hf_icd_combined.csv", out_dir=save_path, dataset_tag="combined", vine=i, cops = cop_4)
#### change the dataset path and dataset_tag 


##### print the word version of the copula structure
##### use it when you save the copula results and want to load the saved results
def print_vine_structure(a, p, c):
    """
    Print a human-readable summary of the complete vine copula structure.

    Iterates over every tree in the vine (from tree 1 to tree d-1) and
    prints each pair-copula edge in the format:
        <conditioned vars> | <conditioning set>  --->  <copula family> : parameters = <params>

    For tree 1 (unconditional pairs) no conditioning set is shown.
    Useful for inspecting a previously saved vine after reloading its
    pickle file.

    Arguments:
        a (np.ndarray): Vine structure matrix (d x d), as returned by
                        `fit_vinecop` or loaded from a pickle file.
        p (np.ndarray): Copula parameter matrix (d-1 x d-1).
        c (np.ndarray): Copula family index matrix (d-1 x d-1).

    Returns:
        None  (output is printed to stdout)
    """
    dimen = a.shape[0]  # number of variables
    for i in range(dimen - 1):  # iterate over each tree level
        print(f"** Tree:  {i + 1}")
        for k in range(dimen - 1 - i):  # iterate over edges in this tree
            ak = a[:, k]  # column k encodes the variable ordering for edge k
            akn = [int(ak[-1 - k]), int(ak[i])]  # conditioned variable pair

            if i == 0:
                # Tree 1: unconditional bivariate copula, no conditioning set
                node_str = f"{akn[0]},{akn[1]}"
            else:
                # Higher trees: include the conditioning set
                conditioning_set = list((ak.astype(int)[:i])[::-1])
                cond_str = ",".join(map(str, conditioning_set))
                node_str = f"{akn[0]},{akn[1]} | {cond_str}"

            # Retrieve the copula name and fitted parameters for this edge
            cop_id = c[i, k]
            name = copula_names.get(cop_id, f"Copula_{cop_id}")
            params = p[i, k]
            print(f"{node_str: <15} --->  {name} : parameters =  {params}")
        print("")  # blank line between trees for readability


In [ ]:
def plotvine_selected(a, plottitle=None, variables=None, savepath=None, tree_to_plot=None):
    """
    Plots the vine structure

    Arguments:
        *a* : The vine tree structure provided as a triangular matrix.
        *plottitle* : title of the plot
        *variables* : list of variable names
        *savepath* : path to save the plot
        *tree_to_plot* : (int) The specific tree number to plot (starts at 1). 
                         If None, plots all trees.

    Returns:
     *plot*
    """
    dimen = a.shape[0] 
    order = pd.DataFrame(
        columns=["node", "l", "r", "tree"]
    ) 

    s = 0
    for i in list(range(dimen - 1)):
        for k in list(range(dimen - 1 - i)):
            ak = a[:, k]
            akn = np.array([ak[-1 - k], ak[i]]).astype(int)
            if i == 0:
                single_row_values = {
                    "node": list(akn),
                    "l": akn[0],
                    "r": akn[1],
                    "tree": i,
                }
            else:
                single_row_values = {
                    "node": list(akn) + ["|"] + list((ak.astype(int)[:i])[::-1]),
                    "l": list(akn),
                    "r": list((ak.astype(int)[:i])[::-1]),
                    "tree": i,
                }

            order.loc[s] = single_row_values
            s = s + 1

    for t in list(range(dimen - 1)):
        orderk = order[order.tree == t].reset_index(drop=True)
        if t == 0:
            orderk["v1"] = orderk.l
            orderk["v2"] = orderk.r
            rhos = []
            v1_1 = []
            v2_1 = []
            cops = []

            for j in range(len(orderk)):
                v1i = int(orderk.v1[j])
                v2i = int(orderk.v2[j])
            locals()["order" + str(t + 1)] = orderk
        else:
            v1k = []
            v2k = []
            for j in range(len(orderk)):
                orderk2 = order[order.tree == t - 1].reset_index(drop=True)
                l = orderk.l[j] + orderk.r[j]
                subnodes = []
                for k in range(len(orderk2)):
                    subnodes.append(sum(1 for item in orderk2.node[k] if item in l))
                subnodes = np.array(subnodes) == len(l) - 1
                orderk2 = orderk2[subnodes].reset_index(drop=True)
                v1k.append(orderk2.node[0])
                v2k.append(orderk2.node[1])
            orderk["v1"] = v1k
            orderk["v2"] = v2k
            locals()["order" + str(t + 1)] = orderk
            
    n = dimen - 1
    
    if tree_to_plot is not None:
        fig, ax = plt.subplots(figsize=((dimen - 1) * 2, (dimen - 1) * 2))
        plot_iterator = [(tree_to_plot, ax)]
    else:
        fig, axes = plt.subplots(dimen - 1, 1, figsize=(n * 2, n * 3))
        plot_iterator = zip(range(1, dimen), axes.flat)

    leg_labels = {}
    if variables != None:
        for i in range(len(variables)):
            leg_labels.update({i: variables[i]})

    for t, ax in plot_iterator: 
        if t == 1:
            orderk = locals()["order" + str(t)]
            edges = list(orderk.node)
            edges = [tuple(sublist) for sublist in edges]
            edge_labels = {
                edge: ",".join(map(str, orderk.node[i])) for i, edge in enumerate(edges)
            }
        elif t == 2:
            orderk = locals()["order" + str(t)]
            edges = [
                (",".join(map(str, orderk.v1[i])), ",".join(map(str, orderk.v2[i])))
                for i in range(len(orderk))
            ]
            edge_labels = {
                edge: ",".join(map(str, orderk.node[i])).replace(",|,", "|")
                for i, edge in enumerate(edges)
            }
        else:
            orderk = locals()["order" + str(t)]
            edges = [
                (
                    ",".join(map(str, orderk.v1[i])).replace(",|,", "|"),
                    ",".join(map(str, orderk.v2[i])).replace(",|,", "|"),
                )
                for i in range(len(orderk))
            ]
            edge_labels = {
                edge: ",".join(map(str, orderk.node[i])).replace(",|,", "|")
                for i, edge in enumerate(edges)
            }

        G = nx.Graph() 
        G.add_edges_from(edges) 
        pos = nx.spring_layout(G)  

        d = nx.degree(G)
        try:
            sizes = len(edges[0][0]) * 400
        except:
            sizes = len(edges[0]) * 200

        nx.draw_networkx_labels(
            G, pos, ax=ax, bbox=dict(facecolor="skyblue"), font_size=13
        ) 

        nx.draw_networkx_edges(
            G,
            pos,
            edge_color="black",
            ax=ax,
        ) 

        nx.draw_networkx_edge_labels(
            G, pos, edge_labels=edge_labels, font_color="black", ax=ax, font_size=12
        ) 

        ax.axis("off") 

        ax.set_title(f"Tree {t}")

    if plottitle != None:
        fig.suptitle(plottitle, fontsize=16, y=0.95) 
    if variables != None:
        plt.text(
            0.9,
            0.5,
            "\n".join([f"{key} : {value}" for key, value in leg_labels.items()]),
            transform=plt.gcf().transFigure,
            fontsize=15,
            verticalalignment="center",
        )

    if savepath != None:
        plt.savefig(savepath, dpi=500, bbox_inches="tight")
    plt.show() 

In [ ]:
##### print the word version of the copula structure
##### use it when you save the copula results and want to load the saved results
def print_vine_structure(a, p, c):
    dimen = a.shape[0]
    for i in range(dimen - 1):
        print(f"** Tree:  {i + 1}")
        for k in range(dimen - 1 - i):
            ak = a[:, k]
            akn = [int(ak[-1 - k]), int(ak[i])]
            
            if i == 0:
                node_str = f"{akn[0]},{akn[1]}"
            else:
                conditioning_set = list((ak.astype(int)[:i])[::-1])
                cond_str = ",".join(map(str, conditioning_set))
                node_str = f"{akn[0]},{akn[1]} | {cond_str}"
            
            cop_id = c[i, k]
            name = copula_names.get(cop_id, f"Copula_{cop_id}")
            params = p[i, k]
            print(f"{node_str: <15} --->  {name} : parameters =  {params}")
        print("")

In [ ]:
gender = 'female'#combined'combined_gender'#
vine_curve = 'C'

with open(f"vine_outputs/4_distribition/balanced_new/vine_{gender}_class1_{vine_curve}.pkl", "rb") as f:
    data = pickle.load(f)
M1 = data["M"]
P1 = data["P"]
C1 = data["C"]

with open(f"vine_outputs/4_distribition/balanced_new/vine_{gender}_class0_{vine_curve}.pkl", "rb") as f:
    data = pickle.load(f)
M0 = data["M"]
P0 = data["P"]
C0 = data["C"]

df_1, df_0 = get_dataset(f"hf_icd_{gender}.csv", "class")

num_tree = 3

plotvine_selected(M1, plottitle=f'{vine_curve}-Vine', variables=list(df_1.columns), savepath=f'vine_outputs/4_distribition/hf_{gender}_{vine_curve}_no{num_tree}.pdf', tree_to_plot=num_tree)
plotvine_selected(M0, plottitle=f'{vine_curve}-Vine', variables=list(df_0.columns), savepath=f'vine_outputs/4_distribition/noncvd_{gender}_{vine_curve}_no{num_tree}.pdf', tree_to_plot=num_tree)


In [ ]:
##### change file path if needed
vine_curve = 'R'
out_dir = 'vine_outputs/4_distribition/balanced_new'

with open(f"vine_outputs/4_distribition/balanced_new/vine_combined_balanced_gender_class_{vine_curve}.pkl", "rb") as f:
    data = pickle.load(f)
M = data["M"]
P = data["P"]
C = data["C"]
# 
datapath = 'hf_icd_gender_balanced.csv'
df = pd.read_csv(datapath) #load data

df = df.drop(columns=['level_0', 'index', 'Unnamed: 0','person_id'], errors='ignore')
df = df.rename(columns = short_items)

num_tree = 1
plotvine_selected(M, plottitle=f'{vine_curve}-Vine', variables=list(df.columns)
                  , savepath=f'vine_outputs/4_distribition/balanced_new/hf_combined_balanced_gender_class_{vine_curve}_{num_tree}.pdf'
                  , tree_to_plot=num_tree)

# print_selected_vine_tree(M, P, C, df.shape[1]-1)

In [ ]:
print_last_vine_tree(M1, P1, C1)
print_last_vine_tree(M0, P0, C0)